# Modul 3: Pemulusan Eksponensial & Holt-Winters
**Mata Kuliah:** Analisis Deret Waktu dan Peramalan (TEE-412) — S1 Teknik Elektro FT UNIB  
**Sub-CPMK 3 (Minggu 3):** Menerapkan *Simple Exponential Smoothing* (SES), *Holt's Linear Trend*, dan *Holt-Winters* (aditif & multiplikatif) untuk *Short-Term Load Forecasting* (STLF).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams['axes.grid'] = True

def eval_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1e-6))) * 100
    wape = np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100
    return {'MAE': mae, 'RMSE': rmse, 'MAPE (%)': mape, 'WAPE (%)': wape}

## 1. Dataset Beban Mingguan & Pemisahan Kronologis Train/Test (24 Jam Terakhir)

In [ ]:
rng = np.random.default_rng(7)
n_hours = 24 * 14  # 14 hari
idx = pd.date_range('2026-02-01', periods=n_hours, freq='h')
t = np.arange(n_hours)
hour = idx.hour

load = (
    500 + 0.15 * t
    + 110 * np.sin(2 * np.pi * (hour - 6) / 24)
    + 65 * np.exp(-((hour - 19) ** 2) / 7)
    + rng.normal(0, 8, n_hours)
)
s_load = pd.Series(load, index=idx, name='load_mw')
train, test = s_load.iloc[:-24], s_load.iloc[-24:]
print(f"Train: {len(train)} jam | Test: {len(test)} jam")

## 2. Komparasi SES vs. Holt-Winters Aditif vs. Holt-Winters Multiplikatif (Damped)

In [ ]:
m_ses = SimpleExpSmoothing(train).fit()
m_hw_add = ExponentialSmoothing(train, seasonal_periods=24, trend='add', seasonal='add').fit()
m_hw_mul = ExponentialSmoothing(
    train, seasonal_periods=24, trend='add', seasonal='mul', damped_trend=True
).fit()

fc_ses = m_ses.forecast(24)
fc_add = m_hw_add.forecast(24)
fc_mul = m_hw_mul.forecast(24)

results = pd.DataFrame({
    'SES': eval_metrics(test, fc_ses),
    'HW Aditif': eval_metrics(test, fc_add),
    'HW Multiplikatif (Damped)': eval_metrics(test, fc_mul),
}).T
print(results.round(3))

plt.figure(figsize=(12, 5))
plt.plot(train.index[-48:], train.iloc[-48:], label='Histori (48 jam terakhir)', color='gray')
plt.plot(test.index, test, label='Aktual (Test)', color='black', linewidth=2.2)
plt.plot(test.index, fc_ses, '--', label='SES')
plt.plot(test.index, fc_add, label='HW Aditif')
plt.plot(test.index, fc_mul, label='HW Multiplikatif Damped')
plt.title('Peramalan Beban Jangka Pendek 24 Jam ke Depan')
plt.ylabel('Beban (MW)')
plt.legend()
plt.show()

## 3. Tugas & Pertanyaan Analisis (C3–C6)
1. **[C3]** Periksa parameter optimal $\alpha, \beta, \gamma$ dari `m_hw_add.params`. Jelaskan makna fisis masing-masing nilai bobot!
2. **[C4]** Simulasikan penurunan beban 20% pada hari terakhir (skenario hari libur nasional) dan amati lonjakan MAPE pada Holt-Winters.
3. **[C5]** Jelaskan mengapa metrik **WAPE** lebih aman dibanding **MAPE** saat meramalkan daya PLTS (yang bernilai 0 kW di malam hari)!